In [20]:
from datasets import load_dataset

dataset = load_dataset("akariasai/PopQA")
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    test: Dataset({
        features: ['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers'],
        num_rows: 14267
    })
})


In [21]:
print(dataset['test'][0])

{'id': 4222362, 'subj': 'George Rankin', 'prop': 'occupation', 'obj': 'politician', 'subj_id': 1850297, 'prop_id': 22, 'obj_id': 2834605, 's_aliases': '["George James Rankin"]', 'o_aliases': '["political leader","political figure","polit.","pol"]', 's_uri': 'http://www.wikidata.org/entity/Q5543720', 'o_uri': 'http://www.wikidata.org/entity/Q82955', 's_wiki_title': 'George Rankin', 'o_wiki_title': 'Politician', 's_pop': 142, 'o_pop': 25692, 'question': "What is George Rankin's occupation?", 'possible_answers': '["politician", "political leader", "political figure", "polit.", "pol"]'}


In [22]:
sample = dataset['test'][0]
for key, value in sample.items():
    print(f"{key}: {value}")

id: 4222362
subj: George Rankin
prop: occupation
obj: politician
subj_id: 1850297
prop_id: 22
obj_id: 2834605
s_aliases: ["George James Rankin"]
o_aliases: ["political leader","political figure","polit.","pol"]
s_uri: http://www.wikidata.org/entity/Q5543720
o_uri: http://www.wikidata.org/entity/Q82955
s_wiki_title: George Rankin
o_wiki_title: Politician
s_pop: 142
o_pop: 25692
question: What is George Rankin's occupation?
possible_answers: ["politician", "political leader", "political figure", "polit.", "pol"]


In [23]:
subset = dataset['test'].select(range(500))
print(len(subset))

500


In [24]:
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:1000]")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

In [25]:
print(wiki[0])

{'id': '12', 'url': 'https://en.wikipedia.org/wiki/Anarchism', 'title': 'Anarchism', 'text': 'Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).\n\nHumans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During

In [134]:
sample2 = wiki[0]
for key, value in sample.items():
    print(f"{key}: {value}")

id: 6250781
subj: Þorsteinn Bachmann
prop: occupation
obj: actor
subj_id: 2822184
prop_id: 22
obj_id: 1114174
s_aliases: ["Porsteinn Bachmann","Thorsteinn Bachmann"]
o_aliases: ["actress","actors","actresses"]
s_uri: http://www.wikidata.org/entity/Q8079549
o_uri: http://www.wikidata.org/entity/Q33999
s_wiki_title: Þorsteinn Bachmann
o_wiki_title: Actor
s_pop: 1252
o_pop: 81374
question: What is Þorsteinn Bachmann's occupation?
possible_answers: ["actor", "actress", "actors", "actresses"]


In [26]:
documents = []

for i, doc in enumerate(wiki):
    text = doc['text']

    chunks = text.split("\n")

    for j, chunk in enumerate(chunks):
        if len(chunk.strip()) > 50:
            documents.append({
                "id": f"P{i}_{j}",
                "text": chunk
            })

In [27]:
print(documents[0])
print(len(documents))
documents = documents[:15000]

{'id': 'P0_0', 'text': 'Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).'}
65385


In [28]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
texts = [doc["text"] for doc in documents]

embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [32]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [33]:
queries = [
    "Who is the founder of anarchism?",
    "Who wrote Harry Potter?",
    "What is the capital of France?"
]

for query in queries:
    print(f"\nQuery: {query}")

    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), k=5)

    for idx in I[0]:
        print(documents[idx]["text"])
        print("-----")


Query: Who is the founder of anarchism?
 Anarchy Archives – an online research center on the history and theory of anarchism.
-----
Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).
-----
The etymological origin of anarchism is from the Ancient Greek anarkhia, meaning "without a ruler", composed of the prefix an- ("without") and the word arkhos ("leader" or "ruler"). The suffix -ism denotes the ideological current that favours anarchy. Anarchism appears in English from 16

In [80]:
def is_relevant(passage, answers):
    passage = passage.lower()

    for ans in answers:
        ans = ans.lower()

        # take meaningful words only
        words = [w for w in ans.split() if len(w) > 4]

        for w in words:
            if w in passage:
                return True

    return False

In [89]:
subset_eval = subset.select(range(100))

In [90]:
k_values = [1, 3, 5]

recall = {k: 0 for k in k_values}
precision = {k: 0 for k in k_values}
mrr_total = 0

for sample in subset_eval:
    query = sample["question"]
    answers = sample["possible_answers"]

    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), k=5)

    retrieved = [documents[idx]["text"] for idx in I[0]]

    # Check relevance
    relevance = [is_relevant(p, answers) for p in retrieved]

    # Recall & Precision
    for k in k_values:
        rel_k = relevance[:k]

        recall[k] += int(any(rel_k))
        precision[k] += sum(rel_k) / k

    # MRR
    rr = 0
    for rank, rel in enumerate(relevance, start=1):
        if rel:
            rr = 1 / rank
            break
    mrr_total += rr

In [91]:
n = len(subset_eval)

for k in k_values:
    recall[k] /= n
    precision[k] /= n

mrr = mrr_total / n

In [94]:
print("Recall:", recall)
print("Precision:", precision)
print("MRR:", mrr)

Recall: {1: 0.0, 3: 0.0, 5: 0.0}
Precision: {1: 0.0, 3: 0.0, 5: 0.0}
MRR: 0.0


In [95]:
def expand_query(query):
    keywords = [
        "biography",
        "who is",
        "definition",
        "meaning",
        "history",
        "details",
        "information"
    ]

    if "who" in query.lower():
        keywords += ["person", "life", "career"]

    if "what" in query.lower():
        keywords += ["explanation", "description"]

    if "where" in query.lower():
        keywords += ["location", "place"]

    return query + " " + " ".join(keywords)

In [96]:
def retrieve(query, k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), k=k)

    return [documents[idx]["text"] for idx in I[0]]

In [97]:
queries = [
    "Who is George Rankin?",
    "Who wrote Harry Potter?",
    "What is the capital of France?"
]

for query in queries:
    expanded = expand_query(query)

    print("\n==============================")
    print("Original:", query)
    print("Expanded:", expanded)

    print("\n--- ORIGINAL RESULTS ---")
    for r in retrieve(query):
        print("-", r[:120])

    print("\n--- EXPANDED RESULTS ---")
    for r in retrieve(expanded):
        print("-", r[:120])


Original: Who is George Rankin?
Expanded: Who is George Rankin? biography who is definition meaning history details information person life career

--- ORIGINAL RESULTS ---
- 1856 – Martin Conway, 1st Baron Conway of Allington, English mountaineer, cartographer, and politician (d. 1937)
-   1943   – Mitchell Melton, American lawyer and politician (d. 2013)
- George MacDonald (1824–1905), author, poet, and theologian born and raised in Huntly.
- Alexander Vovin (2005, 2010, 2017). Formerly an advocate of Altaic (1994, 1995, 1997, 1999, 2000, 2001), later a critic.
- 1755 – Richard Rawlinson, English minister and historian (b. 1690)

--- EXPANDED RESULTS ---
- George MacDonald (1824–1905), author, poet, and theologian born and raised in Huntly.
- 1856 – Martin Conway, 1st Baron Conway of Allington, English mountaineer, cartographer, and politician (d. 1937)
- 1979 – Lord Frederick Windsor, English journalist and financier
-   1943   – Mitchell Melton, American lawyer and politician (d. 

In [99]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc["text"].lower().split() for doc in documents]

bm25 = BM25Okapi(tokenized_corpus)

In [100]:
def bm25_retrieve(query, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    top_k = np.argsort(scores)[::-1][:k]

    return top_k

In [101]:
def dense_retrieve(query, k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), k=k)

    return I[0]

In [104]:
def hybrid_retrieve(query, k=5):
    dense_ids = list(dense_retrieve(query, k))
    bm25_ids = list(bm25_retrieve(query, k))

    scores = {}

    for rank, idx in enumerate(dense_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (rank + 1)

    for rank, idx in enumerate(bm25_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [idx for idx, _ in ranked[:k]]

In [105]:
query = "Who wrote Harry Potter?"

print("\n--- DENSE ---")
for idx in dense_retrieve(query):
    print("-", documents[idx]["text"][:120])

print("\n--- BM25 ---")
for idx in bm25_retrieve(query):
    print("-", documents[idx]["text"][:120])

print("\n--- HYBRID ---")
for idx in hybrid_retrieve(query):
    print("-", documents[idx]["text"][:120])


--- DENSE ---
- 1939 – Alan Ayckbourn, English director and playwright
-  James, James, "Andy Warhol: The Producer as Author", in Allegories of Cinema: American Film in the 1960s (1989), pp. 58
- He also wrote introductions to several collections of classic early 20th century comics, such as Buster Brown, Little Ne
- Aldous Leonard Huxley ( ; 26 July 1894 – 22 November 1963) was an English writer and philosopher. His bibliography spans
- Collected Screenplays. London: Faber & Faber, 2003. .

--- BM25 ---
- 1898 – Harry Edward, Guyanese-English sprinter (d. 1973)
-  Aldous Huxley Collection at the Harry Ransom Center
-  Ann Thwaite Collection of A. A. Milne at the Harry Ransom Center
-  Walter Scott Houston (1912–1993) who wrote the "Deep-Sky Wonders" column in Sky & Telescope magazine for almost 50 year
- 1945 – U.S. President Franklin D. Roosevelt dies in office; Vice President Harry S. Truman becomes President upon Roosev

--- HYBRID ---
- 1939 – Alan Ayckbourn, English director and

In [107]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [108]:
def rerank(query, candidate_ids):
    pairs = [(query, documents[idx]["text"]) for idx in candidate_ids]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(candidate_ids, scores), key=lambda x: x[1], reverse=True)

    return [idx for idx, _ in ranked]

In [110]:
query = "Who wrote Harry Potter?"

candidates = hybrid_retrieve(query, k=10)

reranked_ids = rerank(query, candidates)

print("\n--- BEFORE RERANK ---")
for idx in candidates[:5]:
    print("-", documents[idx]["text"][:120])

print("\n--- AFTER RERANK ---")
for idx in reranked_ids[:5]:
    print("-", documents[idx]["text"][:120])


--- BEFORE RERANK ---
- 1939 – Alan Ayckbourn, English director and playwright
- 1898 – Harry Edward, Guyanese-English sprinter (d. 1973)
-  Aldous Huxley Collection at the Harry Ransom Center
-  James, James, "Andy Warhol: The Producer as Author", in Allegories of Cinema: American Film in the 1960s (1989), pp. 58
- He also wrote introductions to several collections of classic early 20th century comics, such as Buster Brown, Little Ne

--- AFTER RERANK ---
- 1939 – Alan Ayckbourn, English director and playwright
- Aldous Leonard Huxley ( ; 26 July 1894 – 22 November 1963) was an English writer and philosopher. His bibliography spans
- He also wrote introductions to several collections of classic early 20th century comics, such as Buster Brown, Little Ne
-  Ann Thwaite Collection of A. A. Milne at the Harry Ransom Center
-  James, James, "Andy Warhol: The Producer as Author", in Allegories of Cinema: American Film in the 1960s (1989), pp. 58


In [111]:
def get_top_passages(query, k=5):
    candidates = hybrid_retrieve(query, k=10)
    reranked = rerank(query, candidates)
    return reranked[:k]

In [112]:
def build_context(passages_ids):
    context = ""

    for i, idx in enumerate(passages_ids):
        passage = documents[idx]["text"]
        context += f"[P{i}] {passage.strip()}\n\n"

    return context

In [122]:
def build_prompt(query, context):
    return f"""
You are a question answering system.

Answer the question using ONLY the provided passages.

Question:
{query}

Passages:
{context}

Rules:
- Use only the information in the passages
- Cite sources like [P0], [P1]
- If the answer is not found, say: "Insufficient evidence [P0]"
- Do NOT make up information
- Keep the answer concise
"""

In [124]:
from anthropic import Anthropic

client = Anthropic(api_key="")

def generate_answer_claude(query, passages_ids):
    context = build_context(passages_ids)
    prompt = build_prompt(query, context)

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.content[0].text

In [125]:
samples = subset.select(range(10))

for sample in samples:
    query = sample["question"]

    ids = get_top_passages(query)

    print("\n==============================")
    print("QUESTION:", query)

    print("\nPASSAGES:")
    for i, idx in enumerate(ids):
        print(f"[P{i}]", documents[idx]["text"][:120])

    print("\nANSWER:")
    print(generate_answer_claude(query, ids))


QUESTION: What is George Rankin's occupation?

PASSAGES:
[P0] George MacDonald (1824–1905), author, poet, and theologian born and raised in Huntly.
[P1]  George Alcock, discovered several comets and novae.
[P2] 2002 – George Shevelov, Ukrainian-American linguist and philologist (b. 1908)
[P3]  George Orwell's own Nineteen Eighty-Four is a classic dystopian novel about totalitarianism.
[P4]  George Washington University in Washington, D.C. (LL.D.) in 1913

ANSWER:
Insufficient evidence. 

The provided passages contain information about several people named George (MacDonald, Alcock, Shevelov, Orwell, and Washington), but none of them mention anyone named George Rankin or his occupation.

QUESTION: What is John Mayne's occupation?

PASSAGES:
[P0]  Murray, James Erskine, A Summer in the Pyrenees. London: John Macrone, 1837.
[P1]  John Dobson (1915–2014), whose name is associated with the Dobsonian telescope.
[P2] In The Double Clue, Poirot mentions that he was Chief of Police of Brussels

In [ ]:
"""
You are a question answering system.

Answer the question using ONLY the provided passages.

Question:
{query}

Passages:
{context}

Rules:
- Use only the information in the passages
- Cite sources like [P0], [P1]
- If the answer is not found, say: "Insufficient evidence [P0]"
- Do NOT make up information
- Keep the answer concise
"""

In [126]:
def build_reflection_prompt(query, answer, context):
    return f"""
You are evaluating a generated answer.

Question:
{query}

Answer:
{answer}

Passages:
{context}

Evaluate the answer based on:
1. Grounding (is it supported by passages?)
2. Completeness (does it fully answer the question?)
3. Citation quality

Respond with:
- "KEEP" if the answer is correct
- "REVISE" if it can be improved using the same passages
- "RETRIEVE" if more information is needed

Then briefly explain why.
"""

In [131]:
def reflect_answer(query, answer, passages_ids):
    context = build_context(passages_ids)
    prompt = build_reflection_prompt(query, answer, context)

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text

In [132]:
def self_reflective_rag(query):
    ids = get_top_passages(query)

    answer = generate_answer_claude(query, ids)

    critique = reflect_answer(query, answer, ids)

    if "REVISE" in critique:
        answer = generate_answer_claude(query, ids)

    elif "RETRIEVE" in critique:
        ids = hybrid_retrieve(query, k=15)
        ids = rerank(query, ids)[:5]
        answer = generate_answer_claude(query, ids)

    return answer, critique

In [133]:
query = "What is George Rankin's occupation?"

answer, critique = self_reflective_rag(query)

print("ANSWER:", answer)
print("CRITIQUE:", critique)

ANSWER: Insufficient evidence [P0]

The passages provided do not contain any information about George Rankin or his occupation.
CRITIQUE: **KEEP**

**Explanation:**

1. **Grounding**: The answer is properly grounded. The passages provided contain information about several people named George (MacDonald, Alcock, Shevelov, Orwell, Washington), but none of them are "George Rankin." The evaluator correctly identifies that there is no information about George Rankin in the given passages.

2. **Completeness**: The answer fully addresses what can be determined from the available evidence. Given that the passages don't contain relevant information, answering "Insufficient evidence" is the appropriate and complete response.

3. **Citation quality**: The citation [P0] is used correctly to indicate that the passages
